In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS,Chroma # FAISS ,Chroma --> types of VecorDB
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
import spacy
import re
import contractions
from textblob import TextBlob
import os
import string

In [12]:
data = PyPDFLoader("FileHandling.pdf").load()

In [13]:
data = data[0].page_content.lower()

In [14]:
var = string.punctuation
data = re.sub(f'[{var}]',' ',data)

In [15]:
data = re.sub(r'\s{2,}','',data)

In [16]:
nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
tokens

file handlingpg1poornima sfile handlingintroduction to file handlingdef loginpasswordinputusernameinput• data entered in a program is temporary• once the program execution stopsthe data will be lost• to store data permanentlywe use file handlingfile handling allows python programs to store and retrieve data from externalsourcesexternal sourcesexternal sources where data can be stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handling operationsfile handling is mainly used for1reading data from a file2writing data into a file3appending data into an existing file

In [17]:
updated_tokens = [str(token.lemma_) for token in tokens if not token.is_stop] # is_stop--> returns boolean if the word is stopword
data = " ".join(updated_tokens)
print(data)
print(updated_tokens)

file handlingpg1poornima sfile handlingintroduction file handlingdef loginpasswordinputusernameinput• datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe use file handlingfile handling allow python program store retrieve datum externalsourcesexternal sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende datum exist file
['file', 'handlingpg1poornima', 'sfile', 'handlingintroduction', 'file', 'handlingdef', 'loginpasswordinputusernameinput•', 'datum', 'enter', 'program', 'temporary•', 'program', 'execution', 'stopsthe', 'datum', 'lost•', 'store', 'datum', 'permanentlywe', 'use', 'file', 'handlingfile', 'handling', 'allow', 'python', 'program', 'store', 'retrieve', 'datum', 'externalsourcesexternal', 'sourcesexternal', 'source', 'datum', 'stored•', 'text', 'filetxt•', 'csv', 'file•', 'json', 'file•', 'exc

In [18]:
# Fixed size Chuncking
# chunk_size = 150
# s2 = []
# for index in range(0,len(data),chunk_size):
#     chunk = data[index:index+chunk_size]
#     s2.append(chunk)
# print(s2)

In [19]:
# chunk_size = 15
# chunk_overlap = 5
# s2 = []
# for index in range(0,len(data),chunk_size-chunk_overlap):
#     chunk = data[index:index+chunk_size]
#     s2.append(chunk)
# print(s2)

In [20]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 100,
    separators=['\n\n','.','!','\n',' ','']
)
chunks = splitter.create_documents([data])
for chunk in chunks:
    print(chunk.page_content)

file handlingpg1poornima sfile handlingintroduction file handlingdef loginpasswordinputusernameinput• datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe
datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe use file handlingfile handling allow python program store retrieve datum externalsourcesexternal
use file handlingfile handling allow python program store retrieve datum externalsourcesexternal sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file•
sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende
databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende datum exist file


In [21]:
chunks = splitter.create_documents([data])
chunks[0].metadata={'file_name' : 'rag.txt'}
chunks

[Document(metadata={'file_name': 'rag.txt'}, page_content='file handlingpg1poornima sfile handlingintroduction file handlingdef loginpasswordinputusernameinput• datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe'),
 Document(metadata={}, page_content='datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe use file handlingfile handling allow python program store retrieve datum externalsourcesexternal'),
 Document(metadata={}, page_content='use file handlingfile handling allow python program store retrieve datum externalsourcesexternal sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file•'),
 Document(metadata={}, page_content='sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende'),
 Document(metadata={}, page_content='

In [22]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2' # sentence transformer - Encoder only - transformer architecture
)
embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [23]:
vectordb = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

In [24]:
user_query = "What is File handling ?"
r_chunks = vectordb.similarity_search(user_query)  
r_chunks

[Document(id='2209f91e-7a99-4da3-90c4-46fbc5fa57ce', metadata={}, page_content='databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende datum exist file'),
 Document(id='683b4f80-de19-4c38-8525-eee1ce24e6cf', metadata={'file_name': 'rag.txt'}, page_content='file handlingpg1poornima sfile handlingintroduction file handlingdef loginpasswordinputusernameinput• datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe'),
 Document(id='2bbe5ad9-701e-4c6e-94ae-9e2222d6fbbd', metadata={}, page_content='sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende'),
 Document(id='aa086828-90f7-4c80-aa60-0350f9c08170', metadata={}, page_content='datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe use file handlingfile handling allow python

In [25]:
ret_data = ""
for i in r_chunks:
    ret_data += i.page_content
print(ret_data)

databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende datum exist filefile handlingpg1poornima sfile handlingintroduction file handlingdef loginpasswordinputusernameinput• datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywesourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appendedatum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe use file handlingfile handling allow python program store retrieve datum externalsourcesexternal


In [26]:
def retrieval(query : str,k):
    r_chunk = vectordb.similarity_search(query,k)
    retrieval = [chunk.page_content for chunk in r_chunk]
    ret_text = ' '.join(retrieval)
    return ret_text

user_query = "What is File handling ?"
try:
    retrieval(user_query,3)
except Exception as e:
    print(f"Error : {e}")
finally:
    print(retrieval(user_query,3))

databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende datum exist file file handlingpg1poornima sfile handlingintroduction file handlingdef loginpasswordinputusernameinput• datum enter program temporary• program execution stopsthe datum lost• store datum permanentlywe sourcesexternal source datum stored• text filetxt• csv file• json file• excel file• pickle file• databasefile handle operationsfile handling mainly for1reading datum file2write datum file3appende


In [ ]:
user_query = "What is File Handling?"
def generate_text(query,answer):
    model_gemini = ChatGoogleGenerativeAI(
        model = 'gemini-3.5-flash',
        api_key = os.environ['Gemini_api'],
        temperature = 0
    )
    prompt = f'''
            You're a helpful assistant and structure my given context
            Input : {query}
            Output : {answer}
            Note : 
            1)don't add any extra content.
            2)Whatever there in my {answer} just structure it.         
            Output Format:
            Input : {query}
            Output :  
            '''
    gemini = model_gemini.invoke(prompt)
    print("From Gemini : \n",gemini.content[0]['text'])

generate_text(user_query,retrieval(user_query,3))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


From Gemini : 
 Input : What is File Handling ?
Output : 

**file handling pg1 poornima s**

**Introduction File Handling:**
* def login password input username input
* datum enter program temporary
* program execution stops the datum lost
* store datum permanently we sources external source datum stored:
  • text file txt
  • csv file
  • json file
  • excel file
  • pickle file
  • database

**File Handle Operations (file handling mainly for):**
1. reading datum file
2. write datum file
3. appende datum exist file
